Importing Community Detection Methods

In [40]:
# if using a .ipynb in a folder, this heading is needed before importing
import sys
import os

project_root = os.path.abspath(os.path.join(os.getcwd(), '..'))
sys.path.insert(0, project_root)


# in a .py file, just this import works
from src.poisson_hypergraph import GH # custom hypergraph class
from src.algorithms.simulated_annealing import SimulatedAnnealingApprox
from src.algorithms.gradient_descent import GradientDescent

# generally used packages
import xgi
import csv

Importing a Hypergraph (JSON)

In [41]:
# read hypergraph from json
H = xgi.read_json("../throughput/highschool.json", nodetype=int)

# convert into custom hypergraph format that algorithms run on
g = GH(H, [0, 1], 0, 0)

# since this will take to long to demonstrate our algorithms, I will replace the true data with a synthetic hypergraph, but the process is the same

# generate synthetic hypergraph with model, starting with 26 nodes, some of each label
def generate_graph_26_starting_nodes(true_theta, timesteps):
    true_p, true_q, gamma_nu, gamma_nr, gamma_eu, gamma_er = true_theta
    H = xgi.Hypergraph([[0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25]])
    H.set_node_attributes({0:0,1:0,2:0,3:0,4:0,5:0,6:0,7:0,8:0,9:0,10:0,11:0,12:0,13:1,14:1,15:1,16:1,17:1,18:1,19:1,20:1,21:1,22:1,23:1,24:1,25:1}, name="label")
    g = GH(H, [0,1], true_p, true_q)
    g.add_hyperedge(timesteps, gamma_nu, gamma_nr, gamma_eu, gamma_er)

    return g

# define params for hypergraph generation:
eta_plus = .9 # like-labeled copy param
eta_minus = .1 # unlike-labeled copy param
lambda_plus = 1 # like-labeled external param
lambda_minus = .25 # unlike-labeled external param
mu_plus = .001 # like-labeled novel param
mu_minus = .001 # unlike-labeled novel param

true_theta = [eta_plus, eta_minus, mu_plus, mu_minus, lambda_plus, lambda_minus]

# number of edges added to synthetic hypergraph
timesteps = 100

g = generate_graph_26_starting_nodes(true_theta, 100)

/home/fcataldo/labeled-growth/.venv/lib64/python3.9/site-packages/xgi/readwrite/json.py:106: UserWarning: This function is deprecated in favor of the 'read_hif()' function
  warn("This function is deprecated in favor of the 'read_hif()' function")


Running Community Detection

In [42]:
# Modularity Maximization Implementation
from sklearn.metrics import adjusted_rand_score 
import networkx as nx
import numpy as np
from itertools import combinations
def clique_projection_modularity_maximization_algo(g):
    H = g.H
    G = nx.Graph()
    G.add_nodes_from(H.nodes)

    # Clique projection
    for edge in H.edges.members():
        for u, v in combinations(edge, 2):
            if G.has_edge(u, v):
                G[u][v]["weight"] += 1
            else:
                G.add_edge(u, v, weight=1)

    partition = nx.community.greedy_modularity_communities(G, best_n=2)
    z = np.array([0 if node in partition[0] else 1 for node in G.nodes()])

    print(len(z))

    return z

modularity_labels = clique_projection_modularity_maximization_algo(g)
adjusted_rand_score(g.get_labels(), modularity_labels)

26


0.0

In [43]:
# first, declare a parameter set for the community detection algorithm
# in my code, frequently titled "true_theta"
eta_plus = .9 # like-labeled copy param
eta_minus = .1 # unlike-labeled copy param
lambda_plus = 1 # like-labeled external param
lambda_minus = .25 # unlike-labeled external param
mu_plus = .001 # like-labeled novel param
mu_minus = .001 # unlike-labeled novel param

# combine into format for algorithm
true_theta = [eta_plus, eta_minus, mu_plus, mu_minus, lambda_plus, lambda_minus]

# initialize simualted annealing (uses approximate likelihood)
# Takes a LONG time because of the datastructures this approach initializes
# 100 is the max amount of f edges considered to create each e edge
# specifying no approx imlements no approximation
# novel is default set to False, which means only REAL data has no novel nodes
# can be changed by setting to True
sa = SimulatedAnnealingApprox(g, true_theta, approx=100, novel=False)

sa.step() # calls one step on simulated annealing
# simulated annealing schedule designed to run for 20*n steps, where n is the number of nodes in the hypergraph
# The following code provides an exampe of this, though the file paths are different with a notebook
run = True
if run:
    for step_num in range(len(g.nodes)*20):
        sa.step()

        # save run ari and likelihood at each step, write to csv file
        # with open('./throughput/simulated_annealing_highschool_params.csv', 'a', newline="") as file:
        #     writer = csv.writer(file)
        #     writer.writerows([[int(sys.argv[1]), step_num, sa.likelihoods_per_step[-1], sa.aris_per_step[-1]]])

    # get data from run (after done)
    # save simulated annealing max (SAM) results
    # with open('./throughput/simulated_annealing_highschool_max_new_params.csv', 'a', newline="") as file:
    #     writer = csv.writer(file)

    #     writer.writerows([[int(sys.argv[1]), sa.max_LL, sa.max_LL_corresponding_ari]])

    # save final labels
    # with open('./throughput/simulated_annealing_highschool_labels_new_params.csv', 'a', newline="") as file:
    #     writer = csv.writer(file)
    #     writer.writerows([sa.labels])
    #     writer.writerows([sa.max_LL_labels])
    
    print(sa.aris_per_step[-1])

# the lists of final likelihoods and ari per step can be accessed as follows
# sa.likelihoods_per_step
# sa.aris_per_step


1.0


/home/fcataldo/labeled-growth/.venv/lib64/python3.9/site-packages/scipy/stats/_distn_infrastructure.py:1988: RuntimeWarning: divide by zero encountered in divide
  x = np.asarray((x - loc)/scale, dtype=dtyp)


In [ ]:
# initialize gradient descent

for i in range(10):
    g = generate_graph_26_starting_nodes(true_theta, 100)
    # adam params
    def gradient_descent(g, guessed_theta):
        learning_rate = .01
        momentum = (.9,.99)
        gd = GradientDescent(guessed_theta, g, learning_rate, momentum, novel_nodes=True)

        gd.run(500)

        return gd.label_aris[-1]

    print(gradient_descent(g, true_theta))